# 05a - Nonlinear Reduction and Embeddings

How can we turn 64 pixel measurements into a useful two-dimensional map?

A **map** places each observation in the two **output coordinates** of a dimensionality-reduction method.

PCA uses a linear combination of features to produce shared linear directions. Today we’ll explore representations built around **which observations belong near one another**.

## 1.0 The Numbers Define “Similar”

Before making a map, we need numbers. Suppose our records identify three equipment types:

| Equipment | Code |
| --- | ---: |
| Pump | 1 |
| Fan | 2 |
| Compressor | 3 |

An **encoding** represents an observation numerically. These equipment types are **nominal categories**: their names establish no meaningful order.

**Using these codes, is a pump closer to a fan or a compressor? What makes it closer?**

##### After Discussion

For one coordinate, Euclidean distance is the absolute difference:

$$
d(\text{pump},\text{fan})=|1-2|=1
$$

$$
d(\text{pump},\text{compressor})=|1-3|=2
$$

The codes make the fan closer. We supplied that relationship when we assigned the numbers.

### 1.1 Change the Encoding

**One-hot encoding** gives each category its own column. A 1 marks membership; a 0 marks absence.

In [ ]:
import pandas as pd

equipment = pd.Series(["Pump", "Fan", "Compressor"], name="Equipment")
one_hot = pd.get_dummies(equipment, dtype=int)
one_hot = one_hot.loc[:, ["Pump", "Fan", "Compressor"]]
one_hot.index = pd.Index(equipment, name="Equipment")
display(one_hot)

The pump–fan distance is now

$$
d(\text{pump},\text{fan})=\sqrt{(1-0)^2+(0-1)^2+(0-0)^2}=\sqrt{2}
$$

**Your turn:** Calculate the pump–compressor distance. Which equipment type is now closer to the pump?

##### After Discussion

$$
d(\text{pump},\text{compressor})=\sqrt{(1-0)^2+(0-0)^2+(0-1)^2}=\sqrt{2}
$$

They are equally distant. Every distinct pair differs in exactly two columns.

The equipment stayed the same. Its numerical relationships changed.

### 1.2 What Have We Assumed?

One-hot encoding treats every category mismatch equally. Whether that is useful depends on the task.

Now consider **normal → warning → critical**. These are **ordinal categories**: the order has meaning. Coding them as 0, 1, and 2 also assumes equal gaps between successive conditions.

If we add temperature or vibration, their scales and weights will also affect which records become neighbors.

We'll talk more about encoding methods and how to properly represent categorical variables in a subsequent lecture.

## 2.0 Can Two Coordinates Describe a Bent Sheet?

Even with useful input coordinates, a linear projection like PCA can lose relationships we want to keep. Linear combinations are not always an adequate representation of the data.

A **manifold** is a space that can be described locally using ordinary coordinates of a fixed dimension. Consider a smooth sheet, with two independent directions of movement along it. Bending it through three-dimensional space does not add a third direction along the sheet. A small patch of our sheet is approximately flat, even though the whole sheet bends.

For this lecture we'll keep the following concepts separate:

| Dimension | What It Describes | Our Example |
| --- | --- | --- |
| Ambient | Surrounding measurement space | Three spatial coordinates |
| Intrinsic | Independent local coordinates on the manifold | Two directions along the sheet |
| Output | Coordinates requested from the method | Two PCA scores |

To explore the idea we've constructed a **synthetic dataset** from two coordinates per observation: position along the bend and position across the sheet. These **known intrinsic coordinates** appear in the bottom strip.

We then used them to construct an S-shaped surface in three-dimensional space, then added a little noise. These three **ambient coordinates** are the synthetic measurements PCA receives. PCA produces two **output coordinates**, its component scores.

**Follow the circle and square through the ambient, intrinsic, and output views. Which distinction does PCA lose?**

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.lines import Line2D
from sklearn.datasets import load_digits, make_s_curve
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import pairwise_distances
from tqdm import TqdmWarning

# These demonstrations do not use the optional progress-bar widgets.
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="IProgress not found.*", category=TqdmWarning)
    from umap import UMAP

plt.rcParams.update(
    {"figure.dpi": 110, "font.size": 11, "axes.spines.top": False, "axes.spines.right": False}
)
BLUE, ORANGE, GREEN, PURPLE = "#245c91", "#b65313", "#24755b", "#75529a"
SEED = 7
pd.set_option("display.max_colwidth", None)

# Keep this file beside the notebook when using a downloaded copy.
asset_locations = [
    Path.cwd(),
    Path.cwd() / "content/lectures/05a-nonlinear-reduction-and-embeddings",
]
ASSET_DIR = next((p for p in asset_locations if (p / "05a-text-vectors.npz").exists()), Path.cwd())

In [ ]:
curve_x, curve_position = make_s_curve(n_samples=450, noise=0.03, random_state=17)
curve_pca = PCA(n_components=2, svd_solver="full")
curve_scores = curve_pca.fit_transform(curve_x)
input_separation = pairwise_distances(curve_x)
map_separation = pairwise_distances(curve_scores)
# Select a pair whose separation is strongly reduced by this projection.
loss_ratio = input_separation / (map_separation + 0.05)
np.fill_diagonal(loss_ratio, 0)
curve_pair = np.unravel_index(np.argmax(loss_ratio), loss_ratio.shape)

# Reuse the seed without noise to obtain the synthetic data’s known intrinsic coordinates.
curve_clean, curve_clean_position = make_s_curve(n_samples=450, noise=0.0, random_state=17)
curve_intrinsic = np.column_stack((curve_clean_position, curve_clean[:, 1]))

fig = plt.figure(figsize=(10, 7.8), layout="constrained")
grid = fig.add_gridspec(2, 2, height_ratios=[2, 1])
left = fig.add_subplot(grid[0, 0], projection="3d")
left.scatter(*curve_x.T, c=curve_position, cmap="viridis", s=11, alpha=0.7)
left.set(
    xlabel="Ambient 1",
    ylabel="Ambient 2",
    zlabel="Ambient 3",
    title="Ambient coordinates:\nsynthetic measurements",
)
left.view_init(elev=19, azim=65)
left.set_box_aspect(np.ptp(curve_x, axis=0))
right = fig.add_subplot(grid[0, 1])
right.scatter(*curve_scores.T, c=curve_position, cmap="viridis", s=12, alpha=0.7)
right.set(xlabel="PC1 score", ylabel="PC2 score", title="Output coordinates:\nPCA map")
right.set_aspect("equal", adjustable="datalim")
known = fig.add_subplot(grid[1, :])
known.scatter(*curve_intrinsic.T, c=curve_position, cmap="viridis", s=12, alpha=0.7)
known.set(
    xlabel="Position along the bend",
    ylabel="Position across the sheet",
    title="Known intrinsic coordinates: before bending",
)
known.set_aspect("equal", adjustable="box")

# Highlight one small patch of the ideal sheet in both descriptions.
patch_t, patch_y = np.meshgrid(np.linspace(-0.3, 0.3, 9), np.linspace(0.6, 1.4, 9))
patch_x = np.sin(patch_t)
patch_z = np.sign(patch_t) * (np.cos(patch_t) - 1)
left.plot_surface(patch_x, patch_y, patch_z, color="#efb92e", alpha=0.65, shade=False)
known.fill(
    [-0.3, 0.3, 0.3, -0.3], [0.6, 0.6, 1.4, 1.4], facecolor="#efb92e", edgecolor="black", alpha=0.5
)
known.annotate(
    "Small local patch", (0, 1.4), xytext=(1, 2.05), arrowprops={"arrowstyle": "->"}, fontsize=10
)
for row, marker in zip(curve_pair, ["o", "s"], strict=True):
    left.scatter(*curve_x[row], s=100, marker=marker, facecolors="none", edgecolors="black")
    right.scatter(*curve_scores[row], s=100, marker=marker, facecolors="none", edgecolors="black")
    known.scatter(
        *curve_intrinsic[row], s=100, marker=marker, facecolors="none", edgecolors="black"
    )
fig.suptitle("Color and outlined observations follow the same rows in every view")
plt.show()

##### After Discussion

The marked observations differ in their position across the sheet. That distinction remains in the known intrinsic coordinates, but the observations nearly coincide in PCA’s output coordinates. The surface has intrinsic dimension two, yet these two linear scores lose information we wanted.

### 2.1 Local Relationships

PCA finds directions in the ambient feature space that retain the most variance. Every observation gets output coordinates by linear projection onto those same directions. This approach ignores neighborhood relationships that may be important to consider.

A straight line through space can cut across a bend. A path along the sheet must follow the surface. Short steps between neighbors can help describe that surface, provided the measured neighbors trace it well. Noise, sparse samples, or nearby folds can create misleading connections.

None: rotating or translating all the intrinsic-coordinate points together preserves Euclidean distances and neighbor identities. Their coordinate values change, but those relationships stay the same.

### 2.2 Digits

The [scikit-learn digits dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html) supplies small handwritten images. Each handwritten digit is an **8 × 8 image → 64 pixel intensities**. These 64 features are its **ambient coordinates** for reduction. Every intensity uses the same 0–16 scale.

We will request two **output coordinates** per image. The digits’ **intrinsic dimension** is unknown; choosing a two-dimensional map does not establish it.

We will fit the same 600 images each time. Their recorded digit labels color the results; the fitting methods receive only pixels.

In [ ]:
digits = load_digits()
image_ids = np.sort(np.random.default_rng(17).choice(len(digits.data), 600, replace=False))
X = digits.data[image_ids]
digit_labels = digits.target[image_ids]
images = X.reshape(-1, 8, 8)
anchor = int(np.flatnonzero(image_ids == 0)[0])
print(f"Input: {X.shape[0]} images × {X.shape[1]} pixel measurements")

pixel_preview = pd.DataFrame(
    X[:3, [18, 19, 20, 27, 28, 35]],
    index=pd.Index(image_ids[:3], name="Original image ID"),
    columns=[f"Pixel {p}" for p in [18, 19, 20, 27, 28, 35]],
)
display(pixel_preview)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(7, 2.3), layout="constrained")
for ax, row in zip(axes, range(3), strict=True):
    ax.imshow(images[row], cmap="gray_r", vmin=0, vmax=16)
    ax.set_title(f"Image {image_ids[row]} · label {digit_labels[row]}")
    ax.axis("off")
plt.show()


def plot_digit_maps(coordinates, titles):
    """Display output coordinates; fitting happens in separate, visible cells."""
    fig, axes = plt.subplots(
        1, len(coordinates), figsize=(5 * len(coordinates), 4.8), squeeze=False
    )
    cmap = plt.get_cmap("tab10")
    for ax, points, title in zip(axes.flat, coordinates, titles, strict=True):
        ax.scatter(*points.T, c=cmap(digit_labels), s=18, alpha=0.7)
        ax.scatter(*points[anchor], s=115, facecolors="none", edgecolors="black", linewidths=1.7)
        anchor_on_right = points[anchor, 0] > np.median(points[:, 0])
        ax.annotate(
            "Image 0",
            points[anchor],
            xytext=(-7 if anchor_on_right else 7, 9),
            ha="right" if anchor_on_right else "left",
            textcoords="offset points",
            fontsize=9,
            bbox={"facecolor": "white", "alpha": 0.75, "edgecolor": "none", "pad": 1},
        )
        axis_labels = (
            ("PC1 score", "PC2 score")
            if title.startswith("PCA")
            else ("Output coordinate 1", "Output coordinate 2")
        )
        ax.set(title=title, xlabel=axis_labels[0], ylabel=axis_labels[1])
        ax.set_aspect("equal", adjustable="datalim")
    handles = [
        Line2D([], [], marker="o", linestyle="none", color=cmap(i), label=str(i)) for i in range(10)
    ]
    fig.legend(
        handles=handles,
        title="Recorded digit label",
        ncol=min(10, 5 * len(coordinates)),
        loc="lower center",
        fontsize=9,
    )
    caption = (
        "600 images; ring marks image 0"
        if len(coordinates) == 1
        else "Same 600 images; separate output coordinates; ring marks image 0"
    )
    fig.suptitle(caption, fontsize=12)
    fig.subplots_adjust(bottom=0.30 if len(coordinates) == 1 else 0.23, top=0.82, wspace=0.28)
    plt.show()

Follow **original image ID 0**. The first table shows six of its 64 measurements. Image IDs stay attached to rows as we move from 64 ambient pixel coordinates to two output coordinates.

In [ ]:
def nearest_rows(values, k=10):
    distances = pairwise_distances(values, metric="euclidean")
    np.fill_diagonal(distances, np.inf)
    return np.argsort(distances, axis=1, kind="stable")[:, :k]


input_neighbors = nearest_rows(X)

In [ ]:
def plot_neighbor_strips(lists):
    fig, axes = plt.subplots(len(lists), 11, figsize=(14, 1.9 * len(lists) + 0.6), squeeze=False)
    reference_set = set(input_neighbors[anchor])
    for row_index, (method, neighbors) in enumerate(lists.items()):
        selected = [anchor, *neighbors[anchor]]
        for col, row in enumerate(selected):
            ax = axes[row_index, col]
            ax.imshow(images[row], cmap="gray_r", vmin=0, vmax=16)
            ax.set_xticks([])
            ax.set_yticks([])
            if col == 0:
                caption, border = "Image 0\nReference", "black"
            else:
                status = (
                    "Input"
                    if method.startswith("Input")
                    else ("Kept" if row in reference_set else "New")
                )
                caption = f"Rank {col}\nID {image_ids[row]}\n{status}"
                border = GREEN if row in reference_set else ORANGE
            ax.set_title(caption, fontsize=8)
            for spine in ax.spines.values():
                spine.set_visible(True)
                spine.set_edgecolor(border)
                spine.set_linewidth(1.5)
        axes[row_index, 0].set_ylabel(
            method, fontsize=9, rotation=0, ha="right", va="center", labelpad=12
        )
    fig.suptitle("Image 0 and its ten nearest neighbors in each representation", fontsize=13)
    fig.text(
        0.5,
        0.025,
        "Each neighbor is labeled zero. Kept/New describes image identity.",
        ha="center",
        fontsize=10,
    )
    fig.subplots_adjust(
        left=0.14,
        right=0.99,
        top=0.75 if len(lists) == 1 else 0.86,
        bottom=0.16 if len(lists) == 1 else 0.09,
        hspace=1.05,
        wspace=0.2,
    )
    plt.show()


plot_neighbor_strips({"Input · 64 pixels": input_neighbors})

These are image 0’s ten closest other images by Euclidean pixel distance. We will use this list as a reference.

Now fit PCA. Each returned row contains two output coordinates, the PCA scores, for the same image.

In [ ]:
pca_model = PCA(n_components=2, svd_solver="full")
Z_pca = pca_model.fit_transform(X)
print(f"PCA output coordinates: {Z_pca.shape}")

In [ ]:
display(
    pd.DataFrame(Z_pca[:3], index=image_ids[:3], columns=["Output 1", "Output 2"])
    .rename_axis("Image ID")
    .round(3)
)
plot_digit_maps([Z_pca], ["PCA baseline"])

Several digit labels overlap. Can a map built around neighborhoods keep more of the original neighbors?

## 3.0 t-SNE: Match Similarities

**t-SNE** means **t-distributed stochastic neighbor embedding**. 

An **embedding** represents observations by coordinates.

It keeps the ambient pixel coordinates fixed and searches for output coordinates that represent their similarities.

### 3.1 Turn Distances into Relationship Strengths

Start with four observations: **A = 0, B = 1, C = 3, D = 4.5**.

An **affinity** is a numerical relationship strength. Give a neighbor at distance $d$ Gaussian *weight* $\exp(-d^2/2)$, then divide by the sum of weights around the reference observation to get the *probability*.

From A, B receives weight $\exp(-1/2)\approx0.6065$. C receives $\exp(-9/2)\approx0.0111$.

**Before running:** Which observation should receive most of A’s probability?

In [ ]:
toy_names = np.array(["A", "B", "C", "D"])
toy_x = np.array([[0.0], [1.0], [3.0], [4.5]])
toy_distances = pairwise_distances(toy_x)
toy_weights = np.exp(-(toy_distances**2) / 2)
np.fill_diagonal(toy_weights, 0)
toy_conditional = toy_weights / toy_weights.sum(axis=1, keepdims=True)
display(
    pd.DataFrame(
        {
            "Observation": toy_names[1:],
            "Distance from A": toy_distances[0, 1:],
            "Gaussian weight": toy_weights[0, 1:],
            "Probability from A": toy_conditional[0, 1:],
        }
    )
    .set_index("Observation")
    .round(6)
)

##### After Discussion

B receives about 0.982 of A’s probability. Its distance is much smaller. The three probabilities sum to 1; self-comparisons receive zero weight.

### 3.1.1 Process

The output coordinates iterate. The input observations stay fixed.

Suppose we have 600 digit images. t-SNE gives each image two
provisional output coordinates—600 movable points.

Before moving them, it calculates fixed input similarities, $p_{ij}$,
from distances between the images’ 64 pixel measurements. These
describe which pairs should receive strong relationships in the map.

Then the search repeats:

1. Calculate map similarities. Use the current distances between
   output points to calculate $q_{ij}$.

2. Measure the mismatch. Compare these map similarities with the fixed
   input similarities.

3. Calculate coordinate updates. Determine how moving each point would
   change that mismatch. This uses the objective’s gradient.

4. Move the points. Update their output coordinates, then repeat with
   the new distances.

For example, if images A and B have a strong input similarity but weak
map similarity, the updates tend to bring their output points closer.
Moving A also changes its relationship with every other point, so the
search must balance competing relationships.

What changes: output coordinates → output distances → map similarities
→ next coordinate updates.

### 3.2 Find Output Coordinates That Reflect Those Similarities

The toy calculation fixes each Gaussian width at 1. Actual t-SNE chooses widths locally. **Perplexity** controls their effective neighborhood size: larger values broaden the relationships considered.

t-SNE combines both directions of each relationship into symmetric input probabilities, $p$. Proposed output coordinates define a candidate map and supply a second set of probabilities, $q$.

Map weights use $1/(1+d^2)$ and are normalized across pairs. At distance 3, this weight is 0.1, compared with the Gaussian’s 0.0111. This **heavier tail** helps make room for relationships in a low-dimensional map.

**Compare the two proposed layouts and their probabilities. Which better represents the strong A–B input relationship?**

In [ ]:
toy_p = (toy_conditional + toy_conditional.T) / (2 * len(toy_x))
toy_bad_map = np.column_stack(([0, 3, 1, 4.5], np.zeros(4)))
toy_better_map = np.column_stack((toy_x[:, 0], np.zeros(4)))


def map_probabilities(coords):
    weights = 1 / (1 + pairwise_distances(coords) ** 2)
    np.fill_diagonal(weights, 0)
    return weights / weights.sum()


toy_q_bad = map_probabilities(toy_bad_map)
toy_q_better = map_probabilities(toy_better_map)
fig, axes = plt.subplots(1, 2, figsize=(10, 2.8))
for ax, coords, title in zip(
    axes,
    [toy_bad_map, toy_better_map],
    ["Candidate 1: C separates A and B", "Candidate 2: B is beside A"],
    strict=True,
):
    ax.scatter(*coords.T, s=240, color=[BLUE, BLUE, ORANGE, ORANGE])
    for name, point in zip(toy_names, coords, strict=True):
        ax.annotate(name, point, xytext=(0, 14), textcoords="offset points", ha="center")
    ax.set(xlim=(-0.5, 5), ylim=(-0.5, 0.5), yticks=[], xlabel="Output coordinate 1", title=title)
fig.tight_layout()
plt.show()
display(
    pd.DataFrame(
        [
            {
                "Pair": toy_names[i] + "–" + toy_names[j],
                "Input p": toy_p[i, j],
                "Candidate 1 q": toy_q_bad[i, j],
                "Candidate 2 q": toy_q_better[i, j],
            }
            for i, j in [(0, 1), (0, 2), (2, 3)]
        ]
    )
    .set_index("Pair")
    .round(4)
)

##### After Discussion

Candidate 2 puts A and B together. Its A–B probability is closer to the input probability, and A–C is less overstated. These are constructed candidates, not recorded optimizer steps.

### 3.3 Fit the Image Map

The search repeats: **compare input and map probabilities → adjust output coordinates → compare again**.

Moving one point changes several relationships. Attractive and repulsive influences compete; putting every point together would erase the differences between strong and weak pairs.

`fit_transform(X)` fits the map and returns one row of output coordinates per image. PCA supplies initial positions; t-SNE then adjusts them. The seed makes this run repeatable.

In [ ]:
tsne_model = TSNE(
    n_components=2,
    perplexity=30,
    init="pca",
    random_state=SEED,
    learning_rate="auto",
    max_iter=1000,
    metric="euclidean",
)
Z_tsne = tsne_model.fit_transform(X)
print(f"t-SNE output coordinates: {Z_tsne.shape}")

In [ ]:
display(
    pd.DataFrame(Z_tsne[:3].astype(float), index=image_ids[:3], columns=["Output 1", "Output 2"])
    .rename_axis("Image ID")
    .round(3)
)
plot_digit_maps([Z_tsne], ["t-SNE · perplexity 30"])

The groups are easier to see. These output coordinates locate images in the fitted map; they lack PCA’s feature coefficients and variance ranking.

**Clearer groups raise a question: did the same individual neighbors stay close?**

### 3.4 Change Perplexity and Inspect a Neighbor

Compare perplexities **5 and 30**, keeping the images and other settings fixed.

**Find a group whose shape changes. Does the appearance tell us which setting keeps more original neighbors?**

In [ ]:
tsne_small_model = TSNE(
    n_components=2,
    perplexity=5,
    init="pca",
    random_state=SEED,
    learning_rate="auto",
    max_iter=1000,
    metric="euclidean",
)
Z_tsne_small = tsne_small_model.fit_transform(X)
plot_digit_maps([Z_tsne_small, Z_tsne], ["Perplexity 5", "Perplexity 30"])

Use image IDs in the next two strips. Each row is its own ranked list.

**Identify one input neighbor that leaves the top ten and one new image that enters.**

In [ ]:
tsne_neighbors = nearest_rows(Z_tsne)
plot_neighbor_strips(
    {"Input · 64 pixels": input_neighbors, "t-SNE · perplexity 30": tsne_neighbors}
)

In [ ]:
tsne_small_neighbors = nearest_rows(Z_tsne_small)
tsne_neighbors = nearest_rows(Z_tsne)
for label, rows in [("Perplexity 5", tsne_small_neighbors), ("Perplexity 30", tsne_neighbors)]:
    retained = len(set(input_neighbors[anchor]) & set(rows[anchor]))
    print(f"{label}: {retained} of image 0's 10 input neighbors retained")

##### After Discussion

Image 957 leaves the perplexity-30 top ten; image 1716 enters. Image 1167 remains but moves from rank 1 to rank 3. Retention increases from five to six between the two perplexities for this image. We still need a check across the sample.

## 4.0 UMAP: Build a Graph, Then Lay It Out

**UMAP** means **Uniform Manifold Approximation and Projection**. A **graph** represents observations as nodes and their connections as edges. An edge weight says how strongly two observations connect.

### 4.1 Calculate One Connection

Return to **A = 0, B = 1, C = 3, D = 4.5**. Around each observation, UMAP measures distance relative to a local offset and scale. The nearest-neighbor distance supplies the offset in this example.

For illustration, keep every scale at 1 and use all other observations as neighbors.

A’s nearest neighbor is 1 unit away. C’s is 1.5 units away. Their mutual distance is 3:

$$
u_{A\to C}=\exp(-(3-1))\approx0.1353
$$

$$
u_{C\to A}=\exp(-(3-1.5))\approx0.2231
$$

**Why does the same pair distance produce different connection strengths?**

##### After Discussion

Each observation uses its own local reference distance. A distance of 3 is farther beyond A’s nearest neighbor than beyond C’s.

### 4.2 Combine Local Connections

Combine the two directions into one edge:

$$
w_{AC}=0.1353+0.2231-(0.1353)(0.2231)\approx0.3283
$$

A strong connection in either direction contributes to the combined edge. Each edge lies between 0 and 1; the whole graph need not sum to 1.

In [ ]:
toy_nonself = toy_distances.copy()
np.fill_diagonal(toy_nonself, np.inf)
toy_rho = toy_nonself.min(axis=1)
toy_directed = np.exp(-np.maximum(0, toy_distances - toy_rho[:, None]))
np.fill_diagonal(toy_directed, 0)
toy_graph = toy_directed + toy_directed.T - toy_directed * toy_directed.T
display(pd.DataFrame(toy_graph, index=toy_names, columns=toy_names).round(4))

The fitted method chooses local scales from neighbor distances. **`n_neighbors`** controls how many neighbors contribute to graph construction. Larger values bring broader relationships into the graph.

### 4.3 Represent the Graph in a Map

The graph supplies the target connections. UMAP repeatedly adjusts output coordinates through attraction along weighted edges and repulsion from sampled pairs.

**`min_dist`** changes how map distance corresponds to connection strength. Smaller values allow tighter packing. It is a soft layout control, rather than a minimum distance enforced between every pair.

The left panel draws our calculated graph. On the right, follow the dashed A–C weight to each curve.

**Which setting allows that strength at a larger map distance?**

In [ ]:
from umap.umap_ import find_ab_params

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
toy_graph_positions = np.array([[0, 1], [1, 1], [0, 0], [1, 0]])
for i in range(4):
    for j in range(i + 1, 4):
        axes[0].plot(
            *toy_graph_positions[[i, j]].T, color="#888888", lw=5 * toy_graph[i, j], zorder=1
        )
axes[0].scatter(*toy_graph_positions.T, s=450, color=[BLUE, BLUE, ORANGE, ORANGE], zorder=2)
for name, point in zip(toy_names, toy_graph_positions, strict=True):
    axes[0].text(*point, name, color="white", ha="center", va="center", weight="bold")
axes[0].set(xlim=(-0.3, 1.3), ylim=(-0.3, 1.3), title="Input graph: calculated edge weights")
axes[0].axis("off")
map_distances = np.linspace(0, 3, 300)
toy_target_distances = []
for min_dist, color in [(0.1, BLUE), (0.8, ORANGE)]:
    a, b = find_ab_params(spread=1.0, min_dist=min_dist)
    connection = 1 / (1 + a * map_distances ** (2 * b))
    target_distance = ((1 / toy_graph[0, 2] - 1) / a) ** (1 / (2 * b))
    toy_target_distances.append(target_distance)
    axes[1].plot(map_distances, connection, color=color, label=f"min_dist {min_dist}")
    axes[1].plot(
        [target_distance, target_distance], [0, toy_graph[0, 2]], color=color, linestyle=":"
    )
axes[1].axhline(toy_graph[0, 2], color="#666666", linestyle="--", label="A–C input weight")
axes[1].set(
    xlabel="Distance in the map",
    ylabel="Map connection strength",
    ylim=(0, 1.05),
    title="Layout: represent a fixed connection",
)
axes[1].legend(fontsize=9)
fig.tight_layout()
plt.show()

##### After Discussion

With `min_dist=0.8`, the intersection lies farther right. The same graph connection can be represented at a greater map distance. The full layout balances many edges, so this single-edge comparison does not predict an exact fitted distance.

### 4.4 Fit and Inspect the Graph

`fit_transform(X)` builds the graph from ambient pixel coordinates and returns two output coordinates per image. **The graph stores relationships; the output coordinates store map positions.**

In [ ]:
umap_model = UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    spread=1.0,
    metric="euclidean",
    random_state=SEED,
    n_jobs=1,
)
Z_umap = umap_model.fit_transform(X)
print(f"Input graph: {umap_model.graph_.shape}; output coordinates: {Z_umap.shape}")

In [ ]:
display(
    pd.DataFrame(Z_umap[:3].astype(float), index=image_ids[:3], columns=["Output 1", "Output 2"])
    .rename_axis("Image ID")
    .round(3)
)
plot_digit_maps([Z_umap], ["UMAP · 15 neighbors · min_dist 0.1"])

Image 0’s strongest graph links are below. The linked IDs refer to the original images; a weight near 1 indicates a strong connection under this construction.

In [ ]:
anchor_weights = umap_model.graph_.getrow(anchor).toarray().ravel()
strongest_rows = np.argsort(-anchor_weights, kind="stable")[:5]
display(
    pd.DataFrame(
        {
            "Linked image ID": image_ids[strongest_rows],
            "Connection weight": anchor_weights[strongest_rows],
        }
    )
    .set_index("Linked image ID")
    .round(3)
)

### 4.5 Change Compactness

Keep **15 neighbors** and increase **`min_dist` from 0.1 to 0.8**.

**Compare the zero group and the printed graph difference. How can the layout change while its input connections stay fixed?**

In [ ]:
umap_spread_model = UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.8,
    spread=1.0,
    metric="euclidean",
    random_state=SEED,
    n_jobs=1,
)
Z_umap_spread = umap_spread_model.fit_transform(X)
print(f"Changed input-graph entries: {(umap_model.graph_ != umap_spread_model.graph_).nnz}")
plot_digit_maps([Z_umap, Z_umap_spread], ["min_dist 0.1", "min_dist 0.8"])

In [ ]:
umap_neighbors = nearest_rows(Z_umap)
umap_spread_neighbors = nearest_rows(Z_umap_spread)
for label, rows in [("min_dist 0.1", umap_neighbors), ("min_dist 0.8", umap_spread_neighbors)]:
    retained = len(set(input_neighbors[anchor]) & set(rows[anchor]))
    print(f"{label}: {retained} of image 0's 10 input neighbors retained")
    print(
        "Retained IDs:",
        sorted(image_ids[list(set(input_neighbors[anchor]) & set(rows[anchor]))].tolist()),
    )

##### After Discussion

The graph difference is zero. Changing the map’s distance-to-connection relationship spreads out the zero group. Both maps retain four input neighbors, but 676 is retained only at 0.1 and 464 only at 0.8. Equal counts hide a replacement.

## 5.0 Does a Clearer Map Preserve More?

Same ambient pixel coordinates, three different sets of output coordinates. Colors show recorded digit labels; the ring marks image 0.

**Would you judge these maps by how separate the colored groups look? What evidence would you add?**

In [ ]:
plot_digit_maps(
    [Z_pca, Z_tsne, Z_umap],
    ["PCA", "t-SNE · perplexity 30", "UMAP · 15 neighbors, min_dist 0.1"],
)

Use the actual images to check the neighborhoods. **Kept** means the image also belongs to the input top ten; **New** means it entered the map’s top ten. Follow IDs, not vertical columns.

In [ ]:
pca_neighbors = nearest_rows(Z_pca)
neighbor_lists = {
    "Input · 64 pixels": input_neighbors,
    "PCA": pca_neighbors,
    "t-SNE · perplexity 30": tsne_neighbors,
    "UMAP · min_dist 0.1": umap_neighbors,
}

In [ ]:
plot_neighbor_strips(neighbor_lists)

##### After Discussion

All displayed neighbors are labeled zero, yet the identities change. PCA, t-SNE, and UMAP keep four, six, and four input neighbors. Label agreement can hide a change in which handwriting examples are closest.

### 5.1 Check Every Image

For each image, count how many of its ten input neighbors remain among its ten map neighbors:

$$
\text{retained fraction}=\frac{\text{number of shared neighbor IDs}}{10}
$$

Image 0 in t-SNE retains $6/10=0.60$. Repeat for all 600 images and average.

In [ ]:
comparison_rows = []
for method, neighbors in list(neighbor_lists.items())[1:]:
    retained_counts = np.array(
        [
            len(set(original) & set(mapped))
            for original, mapped in zip(input_neighbors, neighbors, strict=True)
        ]
    )
    comparison_rows.append(
        {
            "Method": method,
            "Image 0: retained out of 10": retained_counts[anchor],
            "600 images: mean retained (%)": 100 * retained_counts.mean() / 10,
        }
    )
neighbor_comparison = pd.DataFrame(comparison_rows).set_index("Method")
display(neighbor_comparison.round(2))

**Which method retains most on average? Does that mean it improves every image?**

##### After Discussion

t-SNE retains most on average here. The average does not describe every image: PCA and UMAP tie for image 0. These percentages measure top-ten overlap under our pixel-distance rule.

### 5.2 Choose for a Purpose

A large gap between map islands is not a calibrated input distance. Our overlap check measures neighbors; it does not establish a number of natural clusters.

| Method | What Guides It? | Output Coordinates | Places Future Observations? |
| --- | --- | --- | --- |
| PCA | Retained variance / linear reconstruction | Scores on shared feature directions | Yes |
| t-SNE | Input similarity probabilities | Fitted layout positions | Not with scikit-learn `TSNE` |
| UMAP | Weighted neighborhood graph | Fitted layout positions | Yes |

A colleague recommends UMAP because its zero group looks compact. You need to **inspect handwriting now** and **place new images next month**.

**Would you choose the same method for both jobs? Use one result above and name one further check.**

##### After Discussion

For this sample, t-SNE is a reasonable first inspection map because it retains more pixel neighbors on average. For placing future images, PCA and UMAP provide `transform`; this t-SNE implementation does not. Test a fitted representation on held-out images and inspect their neighbors before choosing. Compact appearance alone does not settle the choice.

## 6.0 What If the Input Vectors Are Learned?

Compare these reports:

> The pump is vibrating more than usual.

> Unusually strong vibration is coming from the pump.

**How could we recognize their related meaning despite different wording?**

A **sentence encoder** applies learned parameters to turn text into a vector. Training encourages related sentences to receive similar representations.

Our pretrained encoder supplies **384 learned coordinates per report**. For a later dimensionality reduction, these vectors would be the **ambient input coordinates**. They are not known intrinsic coordinates of the text.

### 6.1 Follow Reports into Vectors

**Reports → pretrained encoder → vectors → cosine comparisons → original reports**

We use ten invented maintenance reports and their stored vectors. Keep `05a-text-vectors.npz` beside a downloaded notebook. The folded loading cell shows how the file is read.

In [ ]:
text_asset = ASSET_DIR / "05a-text-vectors.npz"
with np.load(text_asset, allow_pickle=False) as stored:
    reports = stored["texts"].tolist()
    text_vectors = stored["vectors"].copy()
    encoder_name = str(stored["model"])
    encoder_revision = str(stored["revision"])

report_table = pd.DataFrame({"Report ID": np.arange(len(reports)), "Report": reports}).set_index(
    "Report ID"
)
display(report_table)
print(f"Vector matrix: {text_vectors.shape}; encoder: {encoder_name}")
display(
    pd.DataFrame(
        text_vectors[:3, :6],
        index=pd.Index(range(3), name="Report ID"),
        columns=[f"Ambient {i}" for i in range(6)],
    ).round(3)
)

The vector table shows six of the 384 ambient coordinates available for a later reduction. Our similarity comparison uses all 384. The model was already trained before seeing these ten reports. Applying it does not train a new encoder.

### 6.2 Compare Directions

Recall cosine similarity:

$$
s(v_i,v_j)=\frac{v_i^{\mathsf T}v_j}{\lVert v_i\rVert\lVert v_j\rVert}
$$

The stored vectors have unit length. Their dot products therefore equal cosine similarities. The matrix product compares every report with every report.

In [ ]:
row_lengths = np.linalg.norm(text_vectors, axis=1)
print(f"Vector lengths: {row_lengths.min():.6f} to {row_lengths.max():.6f}")
text_similarities = text_vectors @ text_vectors.T

In [ ]:
# Begin with a paraphrase and an unrelated report.
first_comparisons = [1, 9]
display(
    pd.DataFrame(
        {
            "Compared with report 0": [reports[i] for i in first_comparisons],
            "Cosine similarity": text_similarities[0, first_comparisons],
        }
    )
    .set_index("Compared with report 0")
    .round(3)
)

The paraphrase scores **0.871**; the unrelated printer report scores **0.037**. These vectors capture useful relatedness across different wording.

### 6.3 Related to Which Task?

Now read: **“The pump is not vibrating more than usual.”**

Would it help retrieve reports **about pump vibration**? Would it support a claim that **vibration is abnormal**?

**What would a high similarity score tell you for each task?**

In [ ]:
ranked_reports = np.argsort(-text_similarities[0, 1:], kind="stable") + 1
text_ranking = pd.DataFrame(
    {
        "Rank": np.arange(1, len(reports)),
        "Report ID": ranked_reports,
        "Report": [reports[i] for i in ranked_reports],
        "Cosine similarity": text_similarities[0, ranked_reports],
    }
)
display(text_ranking.set_index("Rank").round(3))

##### After Discussion

The negated report ranks first at 0.958. It is relevant to retrieving pump-vibration reports, but it does not assert an abnormal condition. High vector similarity can support retrieval while remaining insufficient for that operational decision.

### 6.4 Two Checks, Two Questions

**Representation:** Do the supplied similarities suit the task? Return to the original text or images.

**Reduction:** Does the smaller representation keep the relationships we supplied? Compare input and map neighbors.

Reducing those 384 ambient coordinates to two output coordinates would add another approximation. It could preserve the negated report’s high similarity perfectly and still fail to support an abnormal-condition claim.

## 7.0 One Choice and One Check

Choose a representation for a task you care about.

1. What relationship should it preserve?
2. What evidence would show whether it did?

On Thursday, we will practice these choices alongside PCA review and homework support. HW2B remains based on the PCA material already covered.

## 8.0 Further Detail and Setup

The [fuller companion notebook](05a-nonlinear-reduction-and-embeddings.ipynb) develops the calculations, assumptions, and method comparisons. Both notebooks use the same image sample and stored text vectors.

Use the course Python environment, including `umap-learn==0.5.12`. In a separate environment lacking UMAP, install that version before the imports and restart the kernel if needed. The checked environment uses NumPy 2.5.2 and scikit-learn 1.9.0; versions can affect fitted coordinates.

The text asset includes the exact sentences, encoder name, and revision; `05a-text-vector-provenance.json` records its provenance. The following optional recipe shows how the vectors were created. It requires Sentence Transformers and a model download; normal execution uses the stored file.

```python
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    revision="1110a243fdf4706b3f48f1d95db1a4f5529b4d41",
)
text_vectors = encoder.encode(reports, normalize_embeddings=True)
```

Reading routes: [PDSH manifold introduction](https://jakevdp.github.io/PythonDataScienceHandbook/05.10-manifold-learning.html), [t-SNE paper](https://www.jmlr.org/papers/volume9/vandermaaten08a/vandermaaten08a.pdf), [UMAP explanation](https://umap-learn.readthedocs.io/en/latest/how_umap_works.html), and [Sentence Transformers similarity workflow](https://www.sbert.net/docs/sentence_transformer/usage/semantic_textual_similarity.html).